# K-Nearest Neighbors (KNN) Classifier — Parkinson's Disease Prediction

Train and evaluate a distance-based K-Nearest Neighbors classifier on standardized clinical features.

## 1. Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")

## 2. Load Dataset & Exclude Identifier Column

In [ ]:
df = pd.read_csv("../DATASET/parkinsons_dataset.csv")
unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)
df = df.drop_duplicates()

X = df.drop(columns=["Diagnosis", "PatientID"], errors="ignore")
y = df["Diagnosis"]

print("Feature matrix X shape:", X.shape)

## 3. Stratified Train-Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Hyperparameter Evaluation ($k = 3, 5, 7, 9$)

In [ ]:
k_values = [3, 5, 7, 9]
k_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    k_scores.append({"k": k, "Accuracy": round(acc, 4), "F1 Score": round(f1, 4)})
    print(f"k={k}: Accuracy = {acc * 100:.2f}%, F1 = {f1 * 100:.2f}%")

df_k = pd.DataFrame(k_scores)
print("\nHyperparameter Tuning Results:\n", df_k)

## 5. Train Final KNN Model with Optimal K ($k = 7$)

In [ ]:
final_k = 7
model = KNeighborsClassifier(n_neighbors=final_k)
model.fit(X_train_scaled, y_train)
print(f"Final KNN model trained with k={final_k}.")

## 6. Predictions & Performance Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall:    {rec * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")
print(f"ROC-AUC:   {roc_auc:.4f}")

## 7. Classification Report & Confusion Matrix

In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", cbar=False,
            xticklabels=["Negative (0)", "Positive (1)"],
            yticklabels=["Negative (0)", "Positive (1)"])
plt.title(f"KNN (k={final_k}) Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()